# Phase 15: Advanced Volatility Prediction Experiments

This notebook compares results from 5 experimental improvements to the Phase 14 baseline model:

1. **Learned HAR Weighting** - Adaptive weights for daily/weekly/monthly volatility components
2. **Forward Volatility Target** - Predicting future (60-day forward) volatility instead of past
3. **Enhanced Model** - Multi-head attention + deeper architecture
4. **Ensemble with HAR-RV** - Blending model predictions with pure HAR-RV
5. **Deeper Volatility Head** - Deeper prediction head network

**Key Finding**: Deeper Vol Head achieved **R² = 0.9315**, beating the Phase 14 baseline (0.9212) by +1.1%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Paths
MODELS_DIR = Path('../models')
DATA_DIR = Path('../data')

print("Phase 15 Results Analysis")
print("=" * 50)

## 1. Load Results

In [ ]:
# Load experiment results
results_df = pd.read_csv(MODELS_DIR / 'phase15_experiment_results.csv')

# Load JSON for detailed results
with open(MODELS_DIR / 'phase15_experiment_results.json', 'r') as f:
    results_json = json.load(f)

print("Experiment Results:")
display(results_df[['experiment', 'n_params', 'test_r2', 'test_rmse', 'test_auc', 'epochs_trained']])

## 2. Results Comparison Table

In [ ]:
# Create comparison table
comparison = results_df[['experiment', 'test_r2', 'test_rmse', 'test_auc']].copy()
comparison.columns = ['Experiment', 'Test R²', 'RMSE', 'AUC']

# Calculate improvement over baseline
baseline_r2 = 0.9212
comparison['Δ vs Baseline'] = (comparison['Test R²'] - baseline_r2) * 100
comparison['Δ vs Baseline'] = comparison['Δ vs Baseline'].apply(lambda x: f"{x:+.2f}%" if not pd.isna(x) else '')

# Format
comparison['Test R²'] = comparison['Test R²'].apply(lambda x: f"{x:.4f}" if not pd.isna(x) else '')
comparison['RMSE'] = comparison['RMSE'].apply(lambda x: f"{x:.4f}" if not pd.isna(x) else '')
comparison['AUC'] = comparison['AUC'].apply(lambda x: f"{x:.4f}" if not pd.isna(x) else '')

# Add nice names
name_map = {
    'phase14_baseline': '📊 Phase 14 Baseline',
    'learned_har_weighting': '🎯 Learned HAR Weighting',
    'forward_vol_target': '🔮 Forward Vol Target',
    'enhanced_model': '🧠 Enhanced Model',
    'ensemble_model_har_rv': '🤝 Ensemble + HAR-RV',
    'deeper_vol_head': '🏆 Deeper Vol Head'
}
comparison['Experiment'] = comparison['Experiment'].map(name_map)

print("\n" + "=" * 70)
print("PHASE 15 EXPERIMENT RESULTS")
print("=" * 70)
display(comparison)

## 3. R² Comparison Bar Chart

In [ ]:
# Create R² comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

experiments = ['Phase 14\nBaseline', 'Learned HAR\nWeighting', 'Forward Vol\nTarget', 
               'Enhanced\nModel', 'Ensemble\n+ HAR-RV', 'Deeper Vol\nHead']
r2_values = results_df['test_r2'].values

colors = ['#6c757d', '#17a2b8', '#ffc107', '#28a745', '#dc3545', '#007bff']

bars = ax.bar(experiments, r2_values, color=colors, edgecolor='black', linewidth=1.5)

# Add baseline reference line
ax.axhline(y=baseline_r2, color='red', linestyle='--', linewidth=2, label=f'Phase 14 Baseline (R²={baseline_r2:.4f})')

# Add target line
ax.axhline(y=0.95, color='green', linestyle=':', linewidth=2, label='Target R² = 0.95')

# Add HAR-RV pure baseline
ax.axhline(y=0.779, color='orange', linestyle='-.', linewidth=2, label='HAR-RV Pure Baseline (R²=0.779)')

# Add value labels on bars
for bar, val in zip(bars, r2_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Phase 15 Experiment Results: Test R² Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0.75, 1.0)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'phase15_r2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Chart saved to models/phase15_r2_comparison.png")

## 4. Metrics Overview (R², RMSE, AUC)

In [ ]:
# Create multi-metric comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['test_r2', 'test_rmse', 'test_auc']
titles = ['Test R² (Higher is Better)', 'RMSE (Lower is Better)', 'Direction AUC']
colors = ['#007bff', '#dc3545', '#28a745']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    values = results_df[metric].values
    valid_mask = ~np.isnan(values)
    
    bars = ax.bar(np.arange(len(experiments))[valid_mask], 
                  values[valid_mask], 
                  color=color, alpha=0.7, edgecolor='black')
    
    ax.set_xticks(np.arange(len(experiments)))
    ax.set_xticklabels(experiments, rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, values[valid_mask])):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Phase 15: Multi-Metric Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'phase15_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Findings Summary

In [ ]:
# Find best model
best_idx = results_df['test_r2'].idxmax()
best_model = results_df.loc[best_idx]

print("=" * 70)
print("KEY FINDINGS - PHASE 15 EXPERIMENTS")
print("=" * 70)

print(f"\n🏆 BEST MODEL: {best_model['experiment']}")
print(f"   Test R²: {best_model['test_r2']:.4f}")
print(f"   RMSE:    {best_model['test_rmse']:.4f}")
print(f"   AUC:     {best_model['test_auc']:.4f}")

print(f"\n📊 Improvement over Phase 14 Baseline:")
improvement = (best_model['test_r2'] - baseline_r2) / baseline_r2 * 100
print(f"   +{improvement:.2f}% relative improvement")
print(f"   {best_model['test_r2'] - baseline_r2:.4f} absolute R² increase")

print(f"\n📈 Improvement over HAR-RV Pure Baseline (0.779):")
har_improvement = (best_model['test_r2'] - 0.779) / 0.779 * 100
print(f"   +{har_improvement:.1f}% relative improvement")

print("\n" + "-" * 70)
print("EXPERIMENT RANKINGS (by Test R²):")
print("-" * 70)
rankings = results_df.sort_values('test_r2', ascending=False)[['experiment', 'test_r2', 'test_rmse']]
for i, (_, row) in enumerate(rankings.iterrows(), 1):
    emoji = '🥇' if i == 1 else '🥈' if i == 2 else '🥉' if i == 3 else f'{i}.'
    print(f"  {emoji} {row['experiment']:30s} R²={row['test_r2']:.4f}  RMSE={row['test_rmse']:.4f}")

## 6. Experiment Insights

In [ ]:
print("=" * 70)
print("EXPERIMENT INSIGHTS")
print("=" * 70)

insights = {
    'Deeper Vol Head': {
        'r2': 0.9315,
        'insight': 'BEST RESULT! Adding depth to the volatility prediction head improved performance. '
                   'The deeper network can learn more complex patterns in the fused representations.',
        'why': 'More capacity in the final prediction layers helps capture non-linear volatility dynamics.'
    },
    'Learned HAR Weighting': {
        'r2': 0.9264,
        'insight': 'Second best. Adaptive weighting of HAR-RV components based on VIX regime shows promise. '
                   'Final weights converged to ~[0.34, 0.28, 0.38] favoring monthly component.',
        'why': 'Regime-aware combination of daily/weekly/monthly RV captures volatility clustering.'
    },
    'Enhanced Model': {
        'r2': 0.9240,
        'insight': 'Multi-head attention for cross-modality fusion provides modest gains. '
                   'The attention mechanism helps focus on relevant modalities.',
        'why': 'Attention allows the model to dynamically weight different input sources.'
    },
    'Ensemble + HAR-RV': {
        'r2': 0.9072,
        'insight': 'Surprisingly WORSE than single models. Simple averaging hurts performance. '
                   'The model already incorporates HAR-RV via skip connection.',
        'why': 'Double-counting HAR-RV signal (in model + ensemble) degrades predictions.'
    },
    'Forward Vol Target': {
        'r2': 0.8272,
        'insight': 'Predicting FUTURE volatility is harder but more meaningful. '
                   'R²=0.827 for 60-day forward vol is actually impressive!',
        'why': 'This is the "real" prediction task - forecasting what we cannot yet observe.'
    }
}

for name, data in insights.items():
    print(f"\n📌 {name} (R²={data['r2']:.4f})")
    print(f"   {data['insight']}")
    print(f"   Why: {data['why']}")

## 7. Model Parameters Comparison

In [ ]:
# Parameter efficiency analysis
params_df = results_df[['experiment', 'n_params', 'test_r2']].dropna()

fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(params_df['n_params'] / 1000, params_df['test_r2'], 
                     s=200, c=params_df['test_r2'], cmap='viridis', 
                     edgecolors='black', linewidths=2)

# Add labels
for _, row in params_df.iterrows():
    ax.annotate(row['experiment'].replace('_', '\n'), 
                (row['n_params']/1000, row['test_r2']),
                textcoords="offset points", xytext=(0, 15), 
                ha='center', fontsize=9)

ax.set_xlabel('Parameters (thousands)', fontsize=12)
ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Parameter Efficiency: R² vs Model Size', fontsize=14, fontweight='bold')
ax.axhline(y=baseline_r2, color='red', linestyle='--', alpha=0.7, label='Baseline')
ax.legend()
ax.grid(True, alpha=0.3)

plt.colorbar(scatter, label='Test R²')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'phase15_param_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusions & Recommendations

In [ ]:
print("=" * 70)
print("CONCLUSIONS & RECOMMENDATIONS")
print("=" * 70)

print("""
📊 SUMMARY:
─────────────────────────────────────────────────────────────────────
• Best Model: Deeper Vol Head (R² = 0.9315)
• Improvement over Phase 14: +1.1% (0.9315 vs 0.9212)
• Improvement over HAR-RV: +19.6% (0.9315 vs 0.779)
• Target R² = 0.95 NOT reached, but significant progress made

✅ WHAT WORKED:
─────────────────────────────────────────────────────────────────────
1. Deeper prediction head - More capacity helps
2. Learned HAR weighting - Regime-aware combination
3. Multi-head attention - Cross-modality fusion

❌ WHAT DIDN'T WORK:
─────────────────────────────────────────────────────────────────────
1. Simple ensemble with HAR-RV - Double-counting signal
2. Forward vol prediction - Harder task (but R²=0.827 is impressive!)

🚀 RECOMMENDATIONS FOR FURTHER IMPROVEMENT:
─────────────────────────────────────────────────────────────────────
1. Combine Deeper Vol Head + Learned HAR Weighting
2. Add 10-Q filings for fresher text signals
3. Implement dynamic graph edges with rolling correlations
4. Try larger batch sizes with gradient accumulation
5. Explore different learning rate schedules
6. Consider transformer-based fusion architecture

📁 MODEL FILES:
─────────────────────────────────────────────────────────────────────
• Best model: models/phase15_deeper_vol_head_best.pt
• Results: models/phase15_experiment_results.csv
• Charts: models/phase15_*.png
""")

## 9. Final Comparison vs Phase 14

In [ ]:
# Create final comparison chart
fig, ax = plt.subplots(figsize=(10, 6))

models = ['HAR-RV Pure', 'Phase 14', 'Phase 15\n(Best)']
r2_vals = [0.779, 0.9212, 0.9315]
colors = ['#ffc107', '#17a2b8', '#28a745']

bars = ax.bar(models, r2_vals, color=colors, edgecolor='black', linewidth=2, width=0.6)

# Add value labels
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'R² = {val:.4f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

# Add improvement arrows
ax.annotate('', xy=(1, 0.9212), xytext=(0, 0.779),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(0.5, 0.85, '+18.2%', ha='center', fontsize=12, color='red', fontweight='bold')

ax.annotate('', xy=(2, 0.9315), xytext=(1, 0.9212),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(1.5, 0.93, '+1.1%', ha='center', fontsize=12, color='green', fontweight='bold')

ax.axhline(y=0.95, color='purple', linestyle=':', linewidth=2, label='Target R² = 0.95')

ax.set_ylabel('Test R²', fontsize=14)
ax.set_title('Volatility Prediction: Model Evolution', fontsize=16, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'phase15_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print("🎯 Phase 15 Complete! Best R² = 0.9315 (+1.1% over Phase 14)")
print("=" * 70)